# 🔬 Synthetic Data Generation on Google Colab Pro (Local GPU Inference)

Run the full SDG pipeline in a **Google Colab Pro** GPU runtime using **local model inference**.

**No external API calls.** The model runs entirely on the Colab GPU.

**What this does:**
1. Mounts Google Drive for persistence across disconnects
2. Loads a quantized Llama model directly onto the Colab GPU
3. Monkey-patches `sdg_pipeline.py` to use local inference instead of remote API
4. Generates DPO pairs, niche categories, and nightmare fuel
5. Saves results to Drive

**Runtime required:** Colab Pro (T4 / L4 / A100). T4 (16 GB) handles 4-bit 3B models easily.

In [ ]:
# === Cell 1: Check GPU ===
!nvidia-smi

import torch

if torch.cuda.is_available():
    pass

In [ ]:
# === Cell 2: Mount Drive & Clone Repo ===
from google.colab import drive

drive.mount("/content/drive")

WORKSPACE = "/content/drive/MyDrive/pixelated_sdg"
!mkdir -p {WORKSPACE}

import os

if os.path.exists("/content/pixelated/.git"):
    %cd /content/pixelated
    !git fetch && git pull --no-rebase
else:
    %cd /content
    !git clone https://github.com/daggerstuff/pixelated.git pixelated
    %cd pixelated
    !git submodule init && git submodule update --no-rebase

!ls -la

In [ ]:
# === Cell 3: Install Dependencies ===
!pip install -q transformers accelerate bitsandbytes

import sys

sys.path.insert(0, "/content/pixelated/ai")

In [ ]:
# === Cell 4: Load Local Model onto GPU ===
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# Default: Llama-3.2-3B-Instruct (open, fits in 4-bit on T4)
# Swap to a larger model if you have A100 (40 GB): e.g. meta-llama/Llama-3.1-8B-Instruct
MODEL_ID = "LatitudeGames/Wayfarer-2-12B"  # HF model ID
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=False)
# Wayfarer-2 is 12B params; requires A100 (40 GB) with 4-bit quantization.
# T4 (16 GB) will OOM. If on T4, switch to a smaller model first.
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16,
    load_in_4bit=True,
    trust_remote_code=False,
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.float16,
    device_map="auto",
)


In [ ]:
# === Cell 5: Monkey-Patch sdg_pipeline for Local Inference ===
# We replace _call_nemo (remote HTTP call) with a local GPU inference call.
# All prompt construction, validation, provenance, and file I/O stay exactly the same.

import training.sdg_pipeline as sp


# Disable rate-limit throttle — no need for local inference
def _no_op_throttle(_):
    pass

sp._rate_limit_throttle = _no_op_throttle

def _call_local(prompt, config=None, system_prompt=None, **kwargs):
    """Call the locally loaded model instead of remote NeMo API."""
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})

    outputs = generator(
        messages,
        max_new_tokens=512,
        temperature=0.2,
        do_sample=True,
        return_full_text=False,
    )
    return outputs[0]["generated_text"]

sp._call_nemo = _call_local

In [ ]:
# === Cell 6: Test Generation ===
# Quick sanity check that local inference works and produces expected JSON.

test_prompt = ("Generate a therapeutic training example. Respond ONLY with valid JSON.\n",
             'Format: {"prompt": "...", "chosen": "...", "rejected": "..."}')
test_out = _call_local(test_prompt[0] + test_prompt[1])

In [ ]:
# === Cell 7: Configuration ===
OUTPUT_DIR = Path(WORKSPACE) / "generated"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Adjust based on GPU speed and session length
# T4 ~ 10-20 sec/sample  =>  360-720 samples/hour
# A100 ~ 3-5 sec/sample  =>  1200-2000 samples/hour
TARGET_COUNT   = 500       # per category
MAX_ITERATIONS = 3000
CLIP_VALIDITY  = 0.0

CATEGORIES = [
    "somatic_therapy", "attachment_disorders", "personality_disorders",
    "dissociation", "complicated_grief", "eating_disorders",
    "ocd_intrusive_thoughts", "narcissistic_abuse_recovery",
    "neurodivergent_mental_health", "cultural_religious_contexts"
]
GENERATE_DPO = True
GENERATE_NF  = True


In [ ]:
# === Cell 8: Generate Niche Categories ===
import importlib

importlib.reload(sp)  # ensure fresh state after monkey patch

parser = sp.build_parser()

def gen_category(category):
    out = OUTPUT_DIR / f"{category}.jsonl"
    existing = sum(1 for _ in open(out)) if out.exists() else 0
    need = max(0, TARGET_COUNT - existing)
    if need <= 0:
        return {"category": category, "added": 0, "total": existing}
    args = parser.parse_args([
        "--scenario", "niche_category", "--category", category,
        "--target_count", str(need), "--max_iterations", str(MAX_ITERATIONS),
        "--nemo_endpoint", "http://localhost:9999",  # dummy — local inference ignores this
        "--nemo_api_key", "local", "--nemo_model", MODEL_ID,
        "--nemo_timeout", "120", "--nemo_min_call_interval", "0.1",
        "--min_clinical_validity", str(CLIP_VALIDITY), "--output_path", str(out)])
    sp.run_sdg(args)
    final = sum(1 for _ in open(out)) if out.exists() else 0
    return {"category": category, "added": final-existing, "total": final}

results = []
for cat in CATEGORIES:
    results.append(gen_category(cat))
    # Colab Pro sessions can disconnect; saving to Drive persists across reconnects

for _r in results:
    pass

In [ ]:
# === Cell 9: Generate DPO Pairs ===
if GENERATE_DPO:
    out = OUTPUT_DIR / "dpo_pairs.jsonl"
    existing = sum(1 for _ in open(out)) if out.exists() else 0
    need = max(0, TARGET_COUNT*2 - existing)
    if need > 0:
        args = parser.parse_args([
            "--scenario", "dpo_preference_pairs", "--target_count", str(need),
            "--max_iterations", str(MAX_ITERATIONS),
            "--nemo_endpoint", "http://localhost:9999",
            "--nemo_api_key", "local", "--nemo_model", MODEL_ID,
            "--nemo_timeout", "120", "--nemo_min_call_interval", "0.1",
            "--min_clinical_validity", str(CLIP_VALIDITY), "--output_path", str(out)])
        sp.run_sdg(args)
        final = sum(1 for _ in open(out)) if out.exists() else 0

In [ ]:
# === Cell 10: Generate Nightmare Fuel ===
if GENERATE_NF:
    out = OUTPUT_DIR / "nightmare_fuel.jsonl"
    existing = sum(1 for _ in open(out)) if out.exists() else 0
    need = max(0, TARGET_COUNT - existing)
    if need > 0:
        args = parser.parse_args([
            "--scenario", "nightmare_fuel", "--target_count", str(need),
            "--max_iterations", str(MAX_ITERATIONS),
            "--nemo_endpoint", "http://localhost:9999",
            "--nemo_api_key", "local", "--nemo_model", MODEL_ID,
            "--nemo_timeout", "120", "--nemo_min_call_interval", "0.1",
            "--min_clinical_validity", str(CLIP_VALIDITY), "--output_path", str(out)])
        sp.run_sdg(args)
        final = sum(1 for _ in open(out)) if out.exists() else 0

In [ ]:
# === Cell 11: Summary ===
total = 0
for f in OUTPUT_DIR.glob("*.jsonl"):
    if f.name.startswith("_"): continue
    n = sum(1 for _ in open(f))
    total += n

sample_files = [f for f in OUTPUT_DIR.glob("*.jsonl") if not f.name.startswith("_")]
if sample_files:
    with open(sample_files[0]) as f:
        obj = json.loads(next(f))

In [ ]:
# === Cell 12: Optional — Merge into ChatML ===
merge_script = "/content/pixelated/ai/training/merge_final_dataset.py"
merged_dir = Path(WORKSPACE) / "merged_chatml"
!PYTHONPATH=/content/pixelated/ai uv run --active python {merge_script} \
    --source_dirs {OUTPUT_DIR} \
    --output_dir {merged_dir}